In [1]:
import pandas as pd
import numpy as np

from faker import Faker
import random
import os

fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os

fake = Faker('pt_BR')
np.random.seed(42)
random.seed(42)

# ── CONFIGURAÇÕES ──────────────────────────────────────────

REGIONS    = ['Sul', 'Sudeste', 'Centro-Oeste', 'Norte', 'Nordeste']
CATEGORIES = ['Notebooks', 'Periféricos', 'Smartphones', 'Acessórios', 'Monitores']

PRODUCTS = {
    'Notebooks':    ['Dell Inspiron', 'Lenovo IdeaPad', 'HP Pavilion', 'Asus VivoBook'],
    'Periféricos':  ['Mouse Logitech', 'Teclado Redragon', 'Headset HyperX', 'Webcam Logitech'],
    'Smartphones':  ['Samsung A54', 'Xiaomi 12', 'iPhone 13', 'Motorola Edge'],
    'Acessórios':   ['Cabo USB-C', 'Carregador 65W', 'Suporte Notebook', 'Hub USB'],
    'Monitores':    ['LG 24"', 'Dell 27"', 'Samsung 32"', 'AOC 24"'],
}

BASE_PRICE = {
    'Notebooks':    2800,
    'Periféricos':  180,
    'Smartphones':  1900,
    'Acessórios':   90,
    'Monitores':    1100,
}

# Margem controlada por categoria — garante lucro sempre positivo
# Periféricos com margem baixa é o insight principal
MARGIN_RANGE = {
    'Notebooks':    (0.20, 0.28),
    'Periféricos':  (0.06, 0.14),  # margem baixa — intencional
    'Smartphones':  (0.19, 0.27),
    'Acessórios':   (0.30, 0.42),
    'Monitores':    (0.21, 0.29),
}

MAX_QTY = {
    'Notebooks':    2,
    'Periféricos':  10,
    'Smartphones':  3,
    'Acessórios':   20,
    'Monitores':    3,
}

SEASONALITY = {
    1:  1.10,
    2:  0.85,
    3:  0.90,
    4:  0.95,
    5:  0.90,
    6:  1.00,
    7:  1.05,
    8:  0.95,
    9:  0.95,
    10: 1.00,
    11: 2.20,  # Black Friday
    12: 1.40,  # Natal
}

GOALS_2022 = {
    'Sul':          350000,
    'Sudeste':      550000,
    'Centro-Oeste': 310000,
    'Norte':        220000,
    'Nordeste':     390000,
}

GOAL_SEASONALITY = {
    1:  1.05,
    2:  0.90,
    3:  0.95,
    4:  0.95,
    5:  0.90,
    6:  1.00,
    7:  1.00,
    8:  0.95,
    9:  1.00,
    10: 1.05,
    11: 1.25,
    12: 1.15,
}

GROWTH_2023 = 1.12

# Desconto em passos inteiros de 1%
DISCOUNT_BY_REGION = {
    'Sul':          (2,  10),
    'Sudeste':      (2,  10),
    'Centro-Oeste': (3,  12),
    'Norte':        (5,  15),  # desconto maior — intencional
    'Nordeste':     (3,  12),
}

PERIFERICO_DISCOUNT_BY_REGION = {
    'Sul':          (12, 20),
    'Sudeste':      (12, 20),
    'Centro-Oeste': (15, 22),
    'Norte':        (18, 25),  # desconto ainda maior no Norte
    'Nordeste':     (15, 22),
}

# ── DIM_SELLER ─────────────────────────────────────────────

sellers   = []
seller_id = 1

for region in REGIONS:
    n_sellers = 6 if region == 'Sudeste' else 4
    for _ in range(n_sellers):
        seniority = round(random.uniform(0.5, 8.0), 1)
        sellers.append({
            'vendedor_id':       seller_id,
            'nome_vendedor':     fake.name(),
            'regiao':            region,
            'anos_experiencia':  seniority,
            'equipe':            'Comercial',
        })
        seller_id += 1

dim_seller = pd.DataFrame(sellers)

# ── DIM_METAS ──────────────────────────────────────────────

metas   = []
meta_id = 1

for mes in pd.date_range(start='2022-01-01', end='2023-12-31', freq='MS'):
    year  = mes.year
    month = mes.month
    for regiao, meta_base in GOALS_2022.items():
        meta_ano   = meta_base if year == 2022 else meta_base * GROWTH_2023
        meta_final = round(meta_ano * GOAL_SEASONALITY[month], 2)
        metas.append({
            'meta_id':     meta_id,
            'regiao':      regiao,
            'mes':         mes.strftime('%Y-%m'),
            'meta_mensal': meta_final,
        })
        meta_id += 1

dim_metas = pd.DataFrame(metas)

# ── FACT_SALES ─────────────────────────────────────────────

records = []
sale_id = 1
dates   = pd.date_range(start='2022-01-01', end='2023-12-31', freq='D')

for date in dates:
    month        = date.month
    year         = date.year
    seasonality  = SEASONALITY[month]
    price_factor = 1.0 if year == 2022 else 1.08

    for _, seller in dim_seller.iterrows():

        if seller['regiao'] == 'Norte':
            weights = [0.55, 0.30, 0.10, 0.05]
        elif seller['anos_experiencia'] > 5:
            weights = [0.15, 0.35, 0.30, 0.20]
        elif seller['anos_experiencia'] > 2:
            weights = [0.25, 0.40, 0.25, 0.10]
        else:
            weights = [0.40, 0.40, 0.15, 0.05]

        adjusted_zero = max(0.05, weights[0] / seasonality)
        total_rest    = 1 - adjusted_zero
        scale         = total_rest / sum(weights[1:])
        adj_weights   = [adjusted_zero] + [w * scale for w in weights[1:]]

        n_sales = random.choices([0, 1, 2, 3], weights=adj_weights)[0]

        for _ in range(n_sales):
            category   = random.choice(CATEGORIES)
            produto    = random.choice(PRODUCTS[category])
            base_price = BASE_PRICE[category] * price_factor
            unit_price = round(base_price * random.uniform(1.00, 1.20), 2)

            regiao = seller['regiao']
            if category == 'Periféricos':
                d_min, d_max = PERIFERICO_DISCOUNT_BY_REGION[regiao]
            else:
                d_min, d_max = DISCOUNT_BY_REGION[regiao]

            discount_pct = random.randint(d_min, d_max) / 100
            quantity     = random.randint(1, MAX_QTY[category])
            revenue      = round(unit_price * quantity * (1 - discount_pct), 2)

            # Margem controlada — sem risco de lucro negativo
            margin = random.uniform(*MARGIN_RANGE[category])
            cost   = round(revenue * (1 - margin), 2)
            profit = round(revenue - cost, 2)

            records.append({
                'venda_id':       sale_id,
                'data':           date.strftime('%Y-%m-%d'),
                'vendedor_id':    seller['vendedor_id'],
                'regiao':         seller['regiao'],
                'categoria':      category,
                'produto':        produto,
                'quantidade':     quantity,
                'preco_unitario': unit_price,
                'desconto_pct':   discount_pct,
                'receita':        revenue,
                'custo':          cost,
                'lucro':          profit,
            })
            sale_id += 1

fact_sales = pd.DataFrame(records)

# ── VALIDAÇÃO ──────────────────────────────────────────────

print("=" * 50)
print("✅ Dataset gerado com sucesso!")
print("=" * 50)
print(f"\n📋 fact_sales : {len(fact_sales):>8,} linhas")
print(f"👤 dim_seller : {len(dim_seller):>8,} linhas")
print(f"🎯 dim_metas  : {len(dim_metas):>8,} linhas")

print("\n── Receita por Região ──────────────────────────")
region_summary = (
    fact_sales.groupby('regiao')['receita']
    .sum()
    .sort_values(ascending=False)
    .apply(lambda x: f"R$ {x:,.0f}")
)
print(region_summary.to_string())

print("\n── Margem por Categoria ────────────────────────")
margin_summary = (
    fact_sales.groupby('categoria')[['lucro', 'receita']]
    .sum()
    .assign(margem=lambda x: round(x['lucro'] / x['receita'] * 100, 1))
    ['margem']
    .sort_values()
)
for cat, margin in margin_summary.items():
    print(f"  {cat:<15} {margin}%")

print("\n── Desconto Médio por Região ───────────────────")
discount_summary = (
    fact_sales.groupby('regiao')['desconto_pct']
    .mean()
    .sort_values(ascending=False)
    .apply(lambda x: f"{x*100:.1f}%")
)
print(discount_summary.to_string())

print("\n── Receita 2022 vs 2023 ────────────────────────")
yearly = (
    fact_sales.assign(ano=pd.to_datetime(fact_sales['data']).dt.year)
    .groupby('ano')['receita']
    .sum()
    .apply(lambda x: f"R$ {x:,.0f}")
)
print(yearly.to_string())

print("\n── Checagem de Qualidade ───────────────────────")
print(f"  Nulos fact_sales  : {fact_sales.isnull().sum().sum()}")
print(f"  Nulos dim_seller  : {dim_seller.isnull().sum().sum()}")
print(f"  Nulos dim_metas   : {dim_metas.isnull().sum().sum()}")
print(f"  Lucros negativos  : {(fact_sales['lucro'] < 0).sum()}")
print(f"  Receita negativa  : {(fact_sales['receita'] < 0).sum()}")
print(f"  Vendas duplicadas : {fact_sales['venda_id'].duplicated().sum()}")

# ── EXPORTAR ───────────────────────────────────────────────

data_dir = os.path.join(os.getcwd(), '..', 'data')
os.makedirs(data_dir, exist_ok=True)

fact_sales.to_csv(os.path.join(data_dir, 'fact_sales.csv'),  index=False, encoding='utf-8-sig')
dim_seller.to_csv(os.path.join(data_dir, 'dim_seller.csv'),  index=False, encoding='utf-8-sig')
dim_metas.to_csv(os.path.join(data_dir,  'dim_metas.csv'),   index=False, encoding='utf-8-sig')

print(f"\n📁 Arquivos salvos em: {data_dir}")
print("=" * 50)

✅ Dataset gerado com sucesso!

📋 fact_sales :   18,168 linhas
👤 dim_seller :       22 linhas
🎯 dim_metas  :      120 linhas

── Receita por Região ──────────────────────────
regiao
Sudeste         R$ 14,342,418
Nordeste        R$ 10,101,051
Sul              R$ 8,888,118
Centro-Oeste     R$ 7,633,813
Norte            R$ 4,990,640

── Margem por Categoria ────────────────────────
  Periféricos     10.0%
  Smartphones     23.0%
  Notebooks       24.0%
  Monitores       25.0%
  Acessórios      35.8%

── Desconto Médio por Região ───────────────────
regiao
Norte           12.2%
Centro-Oeste     9.7%
Nordeste         9.7%
Sul              8.1%
Sudeste          8.0%

── Receita 2022 vs 2023 ────────────────────────
ano
2022    R$ 21,927,979
2023    R$ 24,028,062

── Checagem de Qualidade ───────────────────────
  Nulos fact_sales  : 0
  Nulos dim_seller  : 0
  Nulos dim_metas   : 0
  Lucros negativos  : 0
  Receita negativa  : 0
  Vendas duplicadas : 0

📁 Arquivos salvos em: C:\Users\allis\Do